# Baseline 1 — Pure LLM
## GraphRAG Benchmark · Bloomberg Financial News

**Role in benchmark:** The model answers financial questions with **no context** — parametric memory only.  
This is the weakest expected baseline: Mistral-Nemo cannot access article-specific facts it was not trained on.

**Fairness guarantees:**
- Same LLM as the GraphRAG pipeline: `mistral-nemo` via Ollama
- Same 5 000-article Bloomberg subset
- Same 20 questions (saved to `qa_set.json` for Notebook 2 and the GraphRAG pipeline)
- Same RAG Triad evaluation (Faithfulness / Answer Relevance / Context Precision) via `llama3.2:3b` judge

**Memory strategy:** no torch, no transformers, no heavy models — only `requests` and `datasets`.  
Kernel RAM stays under ~1 GB.

## 1. Setup

Minimal dependencies — only what is needed for Ollama calls and dataset streaming.

In [ ]:
!pip install requests datasets tqdm -q
print("Done.")

## 2. Configuration

`N_ARTICLES` must match the GraphRAG pipeline's `CFG['n_articles']` (5 000).  
`N_EVAL` controls how many questions are generated and evaluated — keep at 20 for a balanced benchmark.

In [ ]:
import json, os, re, time, requests
from datasets import load_dataset
from tqdm import tqdm

CFG = {
    "ollama_url"   : "http://localhost:11434",
    "llm_model"    : "mistral-nemo",      # same as GraphRAG pipeline
    "judge_model"  : "llama3.2:3b",       # same as RAGTriadEvaluator in pipeline
    "n_articles"   : 5000,                # must match pipeline
    "n_eval"       : 20,                  # questions generated + evaluated
    "article_max_chars": 1200,            # chars sent to LLM per article
    "qa_set_path"  : "qa_set.json",       # shared with Notebook 2 and GraphRAG
    "results_path" : "baseline_llm_results.json",
    "temperature"  : 0.0,
}

def ollama(model, prompt, timeout=90):
    """Single Ollama call. Returns response string or empty string on error."""
    try:
        r = requests.post(
            f"{CFG['ollama_url']}/api/generate",
            json={"model": model, "prompt": prompt, "stream": False,
                  "options": {"temperature": CFG["temperature"]}},
            timeout=timeout,
        )
        r.raise_for_status()
        return r.json().get("response", "").strip()
    except Exception as e:
        return f"[ERROR: {e}]"

# Quick health check
try:
    r = requests.get(f"{CFG['ollama_url']}/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama up. Models: {models}")
except Exception as e:
    print(f"WARNING: Ollama not reachable — {e}. Start with: ollama serve")

## 3. Dataset Loading

We **stream** the dataset to avoid loading 120K articles into RAM.  
Only the first `N_ARTICLES` are consumed — identical to the pipeline.

In [ ]:
print("Streaming Bloomberg Financial News 120K...")
stream = load_dataset("XJCEO/bloomberg_financial_news_120k", split="train", streaming=True)

articles = []
for i, row in enumerate(stream):
    if i >= CFG["n_articles"]:
        break
    text = row.get("Article") or row.get("article") or ""
    if isinstance(text, str) and len(text.strip()) > 30:
        articles.append({
            "article_id": i,
            "headline"  : str(row.get("Headline", "")),
            "text"      : text,
        })

print(f"Loaded: {len(articles):,} articles")
print(f"Sample headline: {articles[0]['headline']}")

## 4. Shared QA Set Generation

We sample `N_EVAL` articles with a fixed stride across the 5 000-article corpus to ensure **topic diversity**.  
For each article, `mistral-nemo` generates one factual question + reference answer grounded in that article.

**The resulting `qa_set.json` is shared across all three systems** — Pure LLM, Vanilla RAG, and GraphRAG must answer the same questions.

> If `qa_set.json` already exists (e.g. you re-ran this notebook), it is loaded rather than regenerated.

In [ ]:
QA_GEN_PROMPT = """You are a financial QA dataset builder.
Given a Bloomberg news article, write ONE specific factual question that can only be answered by reading this article, and the ideal 2-3 sentence reference answer.

Output ONLY valid JSON — no preamble, no markdown:
{{"question": "...", "reference_answer": "...", "topic": "earnings|ma|personnel|market|policy|other"}}

Headline: {headline}
Article (first 1200 chars):
{text}

JSON:"""

if os.path.exists(CFG["qa_set_path"]):
    with open(CFG["qa_set_path"]) as f:
        qa_set = json.load(f)
    print(f"Loaded existing QA set: {len(qa_set)} questions from {CFG['qa_set_path']}")
else:
    stride = max(1, CFG["n_articles"] // CFG["n_eval"])
    sampled = [articles[i * stride] for i in range(CFG["n_eval"]) if i * stride < len(articles)]

    qa_set = []
    print(f"Generating {len(sampled)} QA pairs...")
    for art in tqdm(sampled, desc="QA generation"):
        raw = ollama(CFG["llm_model"],
                     QA_GEN_PROMPT.format(headline=art["headline"],
                                          text=art["text"][:CFG["article_max_chars"]]))
        try:
            # extract first JSON object from response
            m = re.search(r"\{.*\}", raw, re.DOTALL)
            qa = json.loads(m.group()) if m else {}
            if "question" in qa and "reference_answer" in qa:
                qa_set.append({
                    "qa_id"           : len(qa_set),
                    "article_id"      : art["article_id"],
                    "headline"        : art["headline"],
                    "question"        : qa["question"],
                    "reference_answer": qa["reference_answer"],
                    "topic"           : qa.get("topic", "other"),
                })
                print(f"  [{len(qa_set):2d}] {qa['question'][:70]}")
        except Exception:
            pass

    with open(CFG["qa_set_path"], "w") as f:
        json.dump(qa_set, f, indent=2)
    print(f"\nSaved {len(qa_set)} QA pairs to {CFG['qa_set_path']}")

## 5. Pure LLM Inference

Each question is sent to `mistral-nemo` with **no context** — no article text, no retrieved chunks, no graph.  
The model must rely entirely on parametric memory from pre-training.

In [ ]:
ANSWER_PROMPT = """You are a knowledgeable financial analyst.
Answer the question clearly and concisely in 2-4 sentences.
Base your answer only on your knowledge of financial markets and companies.

Question: {question}

Answer:"""

results = []
t0 = time.time()

print(f"Running Pure LLM on {len(qa_set)} questions (no context)...")
print()

for qa in tqdm(qa_set, desc="Answering"):
    t_start = time.perf_counter()
    answer  = ollama(CFG["llm_model"],
                     ANSWER_PROMPT.format(question=qa["question"]))
    latency = (time.perf_counter() - t_start) * 1000

    results.append({
        "qa_id"           : qa["qa_id"],
        "article_id"      : qa["article_id"],
        "topic"           : qa["topic"],
        "question"        : qa["question"],
        "reference_answer": qa["reference_answer"],
        "predicted_answer": answer,
        "context_used"    : None,
        "latency_ms"      : round(latency, 1),
        "method"          : "pure_llm",
    })

elapsed = time.time() - t0
print(f"\nDone: {len(results)} answers in {elapsed:.0f}s ({elapsed/len(results):.1f}s each)")

# Preview 2
for r in results[:2]:
    print(f"\nQ: {r['question']}")
    print(f"A: {r['predicted_answer'][:200]}")

## 6. RAG Triad Evaluation

**Same evaluator as the GraphRAG pipeline** (`RAGTriadEvaluator` in `graph_generation.ipynb`).  
Three metrics, each 0.0–1.0, judged by `llama3.2:3b`:

| Metric | Definition |
|---|---|
| **Faithfulness** | Every claim in the answer is supported by the context (or general knowledge for Pure LLM) |
| **Answer Relevance** | Answer directly addresses the question |
| **Context Precision** | Fraction of context useful for answering (N/A for Pure LLM → set to 0) |

For Pure LLM: context = `"none"`, so Faithfulness measures hallucination risk against the reference answer, and Context Precision is 0 by definition.

In [ ]:
FAITH_PROMPT = (
    "Rate faithfulness of the answer to the reference (0.0-1.0).\n"
    "Faithfulness = answer makes no claims that contradict the reference.\n\n"
    "Reference: {context}\nAnswer: {answer}\n\n"
    "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}"
)
RELEV_PROMPT = (
    "Rate relevance of the answer to the question (0.0-1.0).\n"
    "1.0 = directly and fully answers the question.\n\n"
    "Question: {query}\nAnswer: {answer}\n\n"
    "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}"
)

def parse_score(raw):
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return float(json.loads(m.group())["score"]) if m else 0.0
    except Exception:
        return 0.0

print(f"Evaluating {len(results)} answers with llama3.2:3b judge...")
print()

for r in tqdm(results, desc="Triad eval"):
    ref = r["reference_answer"]

    faith = parse_score(ollama(CFG["judge_model"],
        FAITH_PROMPT.format(context=ref, answer=r["predicted_answer"])))

    relev = parse_score(ollama(CFG["judge_model"],
        RELEV_PROMPT.format(query=r["question"], answer=r["predicted_answer"])))

    r["faithfulness"]       = round(faith, 3)
    r["answer_relevance"]   = round(relev, 3)
    r["context_precision"]  = 0.0   # no context used
    r["triad_avg"]          = round((faith + relev) / 2, 3)

import numpy as np
print()
print("=" * 55)
print("PURE LLM — BENCHMARK RESULTS")
print("=" * 55)
print(f"{'Metric':25s}  {'Mean':>7}  {'Std':>7}")
print("-" * 42)
for key in ["faithfulness", "answer_relevance", "context_precision", "triad_avg"]:
    vals = [r[key] for r in results]
    print(f"{key:25s}  {np.mean(vals):7.4f}  {np.std(vals):7.4f}")
print(f"{'latency_ms (avg)':25s}  {np.mean([r['latency_ms'] for r in results]):7.1f}")

## 7. Save Results

In [ ]:
summary = {
    "method"             : "Pure LLM (mistral-nemo, no context)",
    "n_questions"        : len(results),
    "mean_faithfulness"  : float(np.mean([r["faithfulness"]      for r in results])),
    "mean_relevance"     : float(np.mean([r["answer_relevance"]  for r in results])),
    "mean_ctx_precision" : 0.0,
    "mean_triad_avg"     : float(np.mean([r["triad_avg"]         for r in results])),
    "mean_latency_ms"    : float(np.mean([r["latency_ms"]        for r in results])),
    "per_question"       : results,
}

with open(CFG["results_path"], "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {CFG['results_path']} ({os.path.getsize(CFG['results_path'])//1024} KB)")
print(f"QA set: {CFG['qa_set_path']}  ← load this in Notebook 2 and the GraphRAG pipeline")

## Summary

| What | Choice | Why |
|------|--------|-----|
| LLM | mistral-nemo via Ollama | Same as GraphRAG pipeline |
| Context | None | Baseline — parametric memory only |
| Dataset | Bloomberg 120K, first 5 000 articles | Same subset as pipeline |
| QA set | 20 questions, stride-sampled for diversity | Shared across all three systems |
| Judge | llama3.2:3b, temp=0 | Same as `RAGTriadEvaluator` in pipeline |
| Memory | No torch/transformers loaded | Kernel stays under 1 GB |

**Expected behaviour vs GraphRAG:** Pure LLM will score well on general financial questions but poorly on event-specific questions about articles that postdate training or were not widely indexed — because it has no access to the dataset.